# معمل التجربة: بناء نظام RAG كلاسيكي باستخدام ChromaDB

في المعمل ده هنمشي خطوة بخطوة مع مسار RAG تقليدي:
1. هنقطع نصوص بسيطة.
2. هنخزنها في قاعدة بيانات ChromaDB محلية.
3. هنحول الفهرس لمسترجع (Retriever).
4. هنسأل سؤال ونخلي نموذج الذكاء الاصطناعي (Groq) يجاوبنا بناءً على المستندات فقط!

### 1. استيراد الأدوات وتجهيز النصوص
يلا نبدأ بتجهيز نصوص تدريبية بسيطة:

In [1]:
import tempfile
import shutil
from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

load_dotenv()
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

sample_texts = [
    Document(page_content="تعلم الآلة (Machine Learning) هو فرع من الذكاء الاصطناعي يركز على بناء أنظمة تتعلم من البيانات دون برمجة صريحة.", metadata={"source": "ml_intro"}),
    Document(page_content="التعلم العميق (Deep Learning) يعتمد على شبكات عصبية اصطناعية متعددة الطبقات تحاكي طريقة عمل الخلايا العصبية لتمثيل البيانات المعقدة.", metadata={"source": "dl_intro"}),
    Document(page_content="محولات الاهتمام (Transformers) تعتمد على آليات الانتباه الذاتي لمعالجة النصوص بشكل متوازي، وهي الأساس لنماذج GPT الحديثة.", metadata={"source": "transformers_intro"}),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
chunks = splitter.split_documents(sample_texts)
print(f"تم تجهيز {len(chunks)} قطعة نصية جاهزة للتخزين!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

تم تجهيز 3 قطعة نصية جاهزة للتخزين!


### 2. التخزين في ChromaDB وبناء المسترجع
تعال نخزن القطع دي في ChromaDB مؤقت ونجرب نسترجع منها:

In [2]:
temp_dir = tempfile.mkdtemp()
vector_store = Chroma.from_documents(chunks, embeddings, persist_directory=temp_dir)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

test_query = "ما هو التعلم العميق وكيف يعمل؟"
found_docs = retriever.invoke(test_query)

print("المستندات المسترجعة:")
for doc in found_docs:
    print(f"- {doc.page_content}")

المستندات المسترجعة:
- تعلم الآلة (Machine Learning) هو فرع من الذكاء الاصطناعي يركز على بناء أنظمة تتعلم من البيانات دون برمجة صريحة.
- محولات الاهتمام (Transformers) تعتمد على آليات الانتباه الذاتي لمعالجة النصوص بشكل متوازي، وهي الأساس لنماذج GPT الحديثة.


### 3. ربط المسترجع بنموذج الذكاء الاصطناعي (Groq)
دلوقتي هندخل النصوص دي في برومبت ونخلي النموذج يجاوبنا:

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)

template = """أنت مساعد ذكي ومفيد. أجب عن السؤال بالاعتماد فقط على السياق التالي:
السياق:
{context}

السؤال:
{question}

الإجابة:"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

answer = rag_chain.invoke("ما هو الفرق الأساسي بين التعلم العميق ونماذج المحولات؟")
print("إجابة النموذج:")
print(answer)

# تنظيف المجلد المؤقت
shutil.rmtree(temp_dir, ignore_errors=True)

إجابة النموذج:
الفرق الأساسي يكمن في المدى والدرجة:

- **التعلم العميق (Deep Learning)** هو فرع من التعلم الآلي يُعنى ببناء نماذج شبكات عصبية ذات طبقات متعددة (أحياناً عشرات أو مئات طبقات). يهدف إلى تمثيل البيانات بطريقة هرمية، ويشمل مجموعة واسعة من الأشكال مثل الشبكات العصبية التلافيفية (CNNs)، الشبكات العصبية المتكررة (RNNs)، وغيرها.

- **نماذج المحولات (Transformers)** هي بنية محددة داخل التعلم العميق. تعتمد على آلية الانتباه الذاتي (self‑attention) التي تسمح للنموذج بالتركيز على أجزاء مختلفة من الإدخال بالتوازي، بدلاً من الاعتماد على التسلسل الزمني كما في RNNs. تُستخدم بشكل أساسي في معالجة اللغة الطبيعية (مثل GPT) وتُعتبر الأساس للعديد من نماذج الذكاء الاصطناعي الحديثة.

إذن، التعلم العميق هو الإطار العام، بينما المحولات هي تقنية معماريّة محددة تُطبّق ضمن هذا الإطار.


### 4. مساحة تجربة حرة
جرب تسأل النموذج سؤال تاني وشوف هيجاوبك إزاي!